In [1]:
"""
Vérifie que 100% des fichiers audio sont à 16 kHz, sur les 3 splits.
Ne lit que l'en-tête de chaque fichier (pas les échantillons) -> rapide
même sur les 71k fichiers de test.

"""

import io
import soundfile as sf
from datasets import Audio

from src.data import load_splits


def check_all_sample_rates(dataset, batch_size=512):
    non_decoded = dataset.cast_column("audio", Audio(decode=False))
    unique_srs = set()
    bad = []

    for i in range(0, len(non_decoded), batch_size):
        batch = non_decoded[i:i + batch_size]
        for j, audio in enumerate(batch["audio"]):
            raw = audio.get("bytes")
            info = sf.info(io.BytesIO(raw)) if raw is not None else sf.info(audio["path"])
            unique_srs.add(info.samplerate)
            if info.samplerate != 16000:
                bad.append((i + j, info.samplerate))

    return unique_srs, bad


if __name__ == "__main__":
    # "test" est déjà en cache chez vous -> pas de téléchargement supplémentaire
    ds = load_splits(splits=("train", "validation", "test"))

    all_ok = True
    for split in ds:
        srs, bad = check_all_sample_rates(ds[split])
        status = "OK" if not bad else "PROBLEME"
        print(f"[{split}] ({len(ds[split])} fichiers) sample rates: {srs} "
              f"| non-16kHz: {len(bad)} -> {status}")
        if bad:
            all_ok = False
            print("   exemples:", bad[:5])

    print("\n" + ("TOUT EST A 16kHz." if all_ok else "ATTENTION: des fichiers ne sont PAS a 16kHz, voir ci-dessus."))

/home/docker/speech-deepfake-detection/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[train] (25380 fichiers) sample rates: {16000} | non-16kHz: 0 -> OK
[validation] (24844 fichiers) sample rates: {16000} | non-16kHz: 0 -> OK
[test] (71237 fichiers) sample rates: {16000} | non-16kHz: 0 -> OK

TOUT EST A 16kHz.
